In [1]:
from proj_utils import *

import os
import pickle
import gc
from os.path import join
from copy import deepcopy
from collections import defaultdict, Counter
from itertools import combinations
from datetime import datetime
from dateutil.relativedelta import relativedelta
import ipywidgets as widgets
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm

import sklearn as sk
from scipy.optimize import linear_sum_assignment
from scipy.optimize import curve_fit
from sklearn.metrics.pairwise import cosine_similarity

import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import mpltern
import ipywidgets as widgets
from ipywidgets import interact, interact_manual

#import bertopic
from proj_utils import *

/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/pyth

### Loading

In [2]:
collection_name = 'Motherjones'
model_name = MODEL_NAMES[collection_name]
threshold = 10

start_date, end_date = DATE_RANGES[collection_name]
num_topics = NUM_TOPICS[collection_name]
#topic_model = (BERTopic.load(join('model', collection_name.lower(), model_name), embedding_model="all-MiniLM-L6-v2"))

In [3]:
# open pickle files
with open(join('article', collection_name.lower(), MODEL_NAMES[collection_name], 'article_dict.pkl'), 'rb') as f:
    article_dict = pickle.load(f)

In [ ]:
# load articles_by_day from feather files and remove articles below threshold

articles_by_day = {}

file_names = os.listdir(join('article', collection_name.lower(), model_name, 'articles_by_day'))
for file_name in file_names:
    day = file_name.split('.')[0]
    print(day)
    articles = pd.read_feather(join('article', collection_name.lower(), model_name, 'articles_by_day', file_name))
    articles_by_day[day] = articles[articles['comment_id'].apply(len) > threshold]

In [ ]:
# merge articles_by_day into articles_by_'month' (not month, but given time interval)

aggregate_days = 30

articles_by_month = dict()
keys = sorted(articles_by_day.keys(), key=lambda x: datetime.strptime(x, '%Y-%m-%d'))

start = datetime.strptime(keys[0], '%Y-%m-%d')
index = 0
while True:
    end = start + relativedelta(days=aggregate_days)
    print(start, end)
    if end > datetime.strptime(keys[-1], '%Y-%m-%d'):
        break
    tmp_list = []
    while True:
        if datetime.strptime(keys[index], '%Y-%m-%d') >= end:
            break
        else:
            tmp_list.append(articles_by_day[keys[index]])
            del articles_by_day[keys[index]]  # for the memory
            index += 1
    if len(tmp_list) > 0:
        articles_by_month[start.strftime('%Y-%m-%d')] = pd.concat(tmp_list)
    start = end

In [ ]:
# load total_mean_embedding_list  and total_nonexist_index_list from pickle
with open(join('article', collection_name.lower(), model_name, f'total_mean_embedding_{aggregate_days}d_list.pkl'), 'rb') as f:
    total_mean_embedding_list = pickle.load(f)
with open(join('article', collection_name.lower(), model_name, f'total_nonexist_index_{aggregate_days}d_list.pkl'), 'rb') as f:
    total_nonexist_index_list = pickle.load(f)
with open(join('article', collection_name.lower(), model_name, f'total_tsne_{aggregate_days}d_list.pkl'), 'rb') as f:
    total_tsne_list = pickle.load(f)

### Preprocessing

In [ ]:
# remove comments after date_cutoff

date_cutoff = 7
topic_by_month = defaultdict(lambda: np.zeros(num_topics+1))
org_list = []
fil_list = []

for article_month in articles_by_month.keys():
    print(article_month)
    articles = articles_by_month[article_month]
    comment_createdAt_list = []
    comment_id_list = []
    comment_topics_list = []
    comment_embeddings_list = []
    topic_freq = Counter({i: 0 for i in range(-1, num_topics)})
    
    for article in articles.itertuples():
        filtered_index = np.where(np.array([(comment_createdAt - article.createdAt).days for comment_createdAt in article.comment_createdAt]) < date_cutoff)[0]
        comment_createdAt_list.append([article.comment_createdAt[i] for i in filtered_index])
        comment_id_list.append([article.comment_id[i] for i in filtered_index])
        comment_topics = [article.comment_topics[i] for i in filtered_index]
        comment_topics_list.append(comment_topics)
        topic_freq.update(comment_topics)
        comment_embeddings_list.append([article.comment_embeddings[i] for i in filtered_index])
        
    # sort topic_freq by key and get values
    topic_freq = [topic_freq[key] for key in sorted(topic_freq.keys())]
    topic_by_month[article_month] = topic_freq
    
    articles_by_month[article_month] = articles_by_month[article_month].assign(comment_id=comment_id_list, comment_topics=comment_topics_list, comment_createdAt=comment_createdAt_list)
    articles_by_month[article_month] = articles_by_month[article_month].assign(comment_id=comment_id_list, comment_topics=comment_topics_list, comment_createdAt=comment_createdAt_list, comment_embeddings=comment_embeddings_list)

In [ ]:
topic_tuple_list_dict = defaultdict(list)
for month in articles_by_month.keys():
    articles = articles_by_month[month]
    for article in articles.itertuples():
        topic_tuple = tuple(sorted(article.topic_num[:3]))
        topic_tuple_list_dict[topic_tuple].append((article._1, article.createdAt.strftime('%Y-%m')))
        
# get the number of elements in each key
num_elements = {key: len(value) for key, value in topic_tuple_list_dict.items()}

# sort the dictionary by the length of the lists
from collections import OrderedDict
sorted_dict = OrderedDict(sorted(num_elements.items(), key=lambda item: item[1], reverse=True))

In [ ]:
# from articles_by_month, get number of comments of each topic by each month, and make it as a widget
article_topics = {}
for month in articles_by_month.keys():
    counter = Counter(np.hstack(articles_by_month[month]['comment_topics']))
    for i in range(-1, num_topics):
        if i not in counter.keys():
            counter[i] = 0
    article_topics[month] = [v[1] for v in sorted(counter.items(), key=lambda x: x[0])]
    
sorted_topic_num = np.vstack(article_topics[month] for month in article_topics.keys()).mean(axis=0).argsort()[::-1]

In [ ]:
# calculate cosine similarity of two vectors
def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_distance_matrix(x):
    distance_matrix = np.zeros((len(x), len(x)))
    for i in range(len(x)):
        for j in range(len(x)):
            #distance_matrix[i, j]= np.sqrt(np.sum((x[i] - x[j])**2))
            if (np.linalg.norm(x[i]) * np.linalg.norm(x[j])) != 0:
                distance_matrix[i, j] = cos_sim(x[i], x[j])
    return distance_matrix

mean_weight_list = []
distance_matrix_list = []
for i in range(len(total_mean_embedding_list)):
    x = total_mean_embedding_list[i]

    # for x, I want to get a distance matrix between all pairs
    distance_matrix = get_distance_matrix(x)
    distance_matrix = np.delete(distance_matrix, total_nonexist_index_list[i], axis=0)
    distance_matrix = np.delete(distance_matrix, total_nonexist_index_list[i], axis=1)
    distance_matrix_list.append(distance_matrix)
    mean_weight = np.sum(distance_matrix, axis=0) / (len(distance_matrix)-1)
    mean_weight_list.append(mean_weight)

In [ ]:
topic_num_list = np.zeros((len(articles_by_month.keys()), num_topics, 3))
constant_list = np.zeros(len(articles_by_month.keys()))

for m, (month, articles) in enumerate(articles_by_month.items()):
    constant_list[m] = len(articles_by_month[month])
    for t in range(num_topics):
        for i in range(3):
            topic_num_list[m][t][i] = len(articles[np.vstack(articles['topic_num'])[:, i] == t])

norm_topic_num_list = topic_num_list / constant_list[:, np.newaxis, np.newaxis]

### Preprocessing (for new aggregate_days)

In [ ]:
## Adaptively constructing total_mean_embedding_list and total_nonexist_index_list, according to the aggregate_days

total_mean_embedding_list= []
total_nonexist_index_list = []
for article_month in articles_by_month.keys():
    print(article_month)
    articles = articles_by_month[article_month]
    embedding_list = [[] for _ in range(num_topics+1)] # including -1
    mean_embedding_list = [[] for _ in range(num_topics+1)] # including -1
    nonexist_index_list = [] 
    for article in articles.itertuples():
        for i in range(num_topics+1):  # including -1  
            comment_index = np.where(np.array(article.comment_topics) == i-1)[0]
            if len(comment_index)>0:
                embedding_list[i].append(article.comment_embeddings[comment_index])
            
    for i in range(num_topics+1): 
        if len(embedding_list[i])>0:
            averaged = np.vstack(embedding_list[i]).mean(axis=0)
        else:
            nonexist_index_list.append(i)
            averaged = np.zeros(384)  # embedding shape
        mean_embedding_list[i] = averaged

    print(nonexist_index_list)
    mean_embedding_list = np.array(mean_embedding_list)
    total_mean_embedding_list.append(mean_embedding_list)
    total_nonexist_index_list.append(nonexist_index_list)
    
total_mean_embedding_list = np.array(total_mean_embedding_list)

# save total_mean_embedding_list and total_nonexist_index_list as pickle
with open(join('article', collection_name.lower(), model_name, f'total_mean_embedding_{aggregate_days}d_list.pkl'), 'wb') as f:
    pickle.dump(total_mean_embedding_list, f)
with open(join('article', collection_name.lower(), model_name, f'total_nonexist_index_{aggregate_days}d_list.pkl'), 'wb') as f:
    pickle.dump(total_nonexist_index_list, f)

In [ ]:
# for each element of total_mean_embedding_list, remove all-zero rows (non-existing topics) and concatenate all of them.
embedding_list_tsne = []

for i, mean_embedding in enumerate(total_mean_embedding_list):
    mean_embedding = np.delete(mean_embedding, total_nonexist_index_list[i], axis=0)
    embedding_list_tsne.append(mean_embedding)
    
embedding_list_tsne = np.concatenate(embedding_list_tsne, axis=0)
X_embedded = sk.manifold.TSNE(n_components=2).fit_transform(embedding_list_tsne)

total_tsne_list = []
previous_list = np.array([[0, 0]] * (num_topics+1)) # including -1
counter = 0
for i in range(len(articles_by_day.keys())):
    print(i)
    tsne_list = []
    nonexist_index_list = deepcopy(total_nonexist_index_list[i])
    if len(nonexist_index_list)==0:
        tsne_list = X_embedded[counter:counter+num_topics+1]
        previous_list = X_embedded[counter:counter+num_topics+1]
        counter+=num_topics+1
    else:
        current_index = nonexist_index_list[0]
        for j in range(num_topics+1):
            if j==current_index:
                tsne_list.append(previous_list[j])
                nonexist_index_list.pop(0)
                if len(nonexist_index_list)>0:
                    current_index = nonexist_index_list[0]
            else:
                tsne_list.append(X_embedded[counter])
                previous_list[j] = X_embedded[counter]
                counter+=1
            
    assert len(tsne_list)==num_topics+1
    total_tsne_list.append(np.array(tsne_list))
    
total_tsne_list = np.array(total_tsne_list)

with open(join('article', collection_name.lower(), model_name, f'total_tsne_{aggregate_days}d_list.pkl'), 'wb') as f:
    pickle.dump(total_tsne_list, f)

## 1. Basic statistics

### A. Number of articles

### A.1. Number of articles with number of comments
* How many article with certain number of comments?

In [ ]:
# draw histogram of number of comments of articles from article_dict

num_articles_with_threshold_comments = sum([len(article_dict[key]['id']) > threshold for key in article_dict.keys()])
total_num_articles = len(article_dict.keys())
percentage = num_articles_with_threshold_comments / total_num_articles * 100
print(f"{percentage:.2f}% of articles have more than {threshold} comments.")

num_comments_list = [len(article_dict[key]['id']) for key in article_dict.keys()]
plt.hist(num_comments_list, bins=500, cumulative=True, density=True)
plt.xlabel('Number of comments')
plt.ylabel('Number of articles')
plt.title('Number of comments per article')
plt.show()

### A.2. Number of articles (threshold)
* How many article (which has more than *THRESHOLD* comments) were posted each month?

In [ ]:
num_articles = [len(articles_by_month[month]) for month in articles_by_month.keys()]
total_articles = sum(num_articles)
df = pd.DataFrame({'Month': list(articles_by_month.keys()), 'Num Articles': num_articles})
df.plot(x='Month', y='Num Articles',  figsize=(10, 7), kind='bar', title=f'Number of Articles per Month (Total : {total_articles})')

### B. Number of comments

#### B.1. Including -1 (non-classified)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

total_frequency.plot(kind='bar', ax=ax, color='blue', alpha=0.5, label='Without threshold (raw)')
ax.set_title('Number of Comments per Month (including -1)')
ax.set_ylabel('Number of Comments')

# get number of comments in articles_by_month
num_comments = []
for month in articles_by_month.keys():
    num_comments.append(articles_by_month[month]['comment_id'].apply(len).sum())
df = pd.DataFrame({'Month': list(articles_by_month.keys()), f'With threshold {threshold} & cutoff {date_cutoff}': num_comments})
df.plot(x='Month', y=f'With threshold {threshold} & cutoff {date_cutoff}',  figsize=(10, 7), kind='bar', title=f'Number of Comments per Month (Total : {sum(num_comments)})', ax=ax)

ax.legend()
plt.show()

#### B.2. Excluding -1 (non-classified)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

total_frequency2.plot(kind='bar', ax=ax, color='blue', alpha=0.5, label='Without threshold (raw)')
ax.set_title('Number of Comments per Month (excluding -1)')
ax.set_ylabel('Number of Comments')

# get number of comments in articles_by_month
num_comments = []
for month in articles_by_month.keys():
    comments = articles_by_month[month]['comment_topics']
    num_list = []
    for comment in comments:
        m1_count = len([topic for topic in comment if topic == -1])
        num_list.append(len(comment) - m1_count)
    num_comments.append(sum(num_list))

df = pd.DataFrame({'Month': list(articles_by_month.keys()), f'With threshold {threshold} & cutoff {date_cutoff}': num_comments})
df.plot(x='Month', y=f'With threshold {threshold} & cutoff {date_cutoff}',  figsize=(10, 7), kind='bar', title=f'Number of Comments per Month (Total : {sum(num_comments)})', ax=ax)

ax.legend()
plt.show()

### C. Title topics
* What is the top 3 topic distribution of title?

#### C.1 Total

In [ ]:
@interact(rank=widgets.IntSlider(min=1, max=3, step=1, value=1), normalize=widgets.Checkbox(value=False, description='Normalize'))
def plot_histogram_by_month(rank, normalize):
    topic_num = np.vstack([np.vstack(articles_by_month[month]['topic_num'].values) for month in articles_by_month.keys()])
    alpha_list = [1 if i==rank-1 else 0.3 for i in range(3)]

    freq1 = plt.hist(topic_num[:, 0], bins=range(int(topic_num[:, 0].min()), int(topic_num[:, 0].max()) + 2, 1), color='r', alpha=alpha_list[0], density=normalize)
    freq2 = plt.hist(topic_num[:, 1], bins=range(int(topic_num[:, 1].min()), int(topic_num[:, 1].max()) + 2, 1), color='g', alpha=alpha_list[1], density=normalize)
    freq3 = plt.hist(topic_num[:, 2], bins=range(int(topic_num[:, 2].min()), int(topic_num[:, 2].max()) + 2, 1), color='b', alpha=alpha_list[2], density=normalize)
    plt.title(f"Histogram of Article Topic Numbers for {start_date.strftime('%Y-%m')} to {end_date.strftime('%Y-%m')}")
    plt.xlabel("Topic Number")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

@interact(rank=widgets.IntSlider(min=1, max=3, step=1, value=1), sort=True, normalize=False)
def plot_histogram_with_curve_fit(rank, sort=True, normalize=False):
    topic_num = np.vstack([np.vstack(articles_by_month[month]['topic_num'].values) for month in articles_by_month.keys()])
    alpha_list = [0.9 if i==rank-1 else 0.3 for i in range(3)]

    freq1 = plt.hist(topic_num[:, 0], bins=range(int(topic_num[:, 0].min()), int(topic_num[:, 0].max()) + 2, 1), color='r', alpha=alpha_list[0], density=normalize)
    freq2 = plt.hist(topic_num[:, 1], bins=range(int(topic_num[:, 1].min()), int(topic_num[:, 1].max()) + 2, 1), color='g', alpha=alpha_list[1], density=normalize)
    freq3 = plt.hist(topic_num[:, 2], bins=range(int(topic_num[:, 2].min()), int(topic_num[:, 2].max()) + 2, 1), color='b', alpha=alpha_list[2], density=normalize)
    
    def func(x, a, b, c):
        return a*np.exp(-b*x) + c

    # sorting freq
    if rank==1:
        freq = freq1
    elif rank==2:
        freq = freq2
    else:
        freq = freq3

    if sort:
        freq = freq[0][np.argsort(freq[0])[::-1]]
    else:
        freq = freq[0]

    # find the best fitting curve for this histogram
    xdata = np.arange(len(freq)-1)+1
    popt, pcov = curve_fit(func, xdata, freq[1:])
    plt.bar(range(len(freq)), freq, alpha=1, color='k')
    plt.plot(xdata, func(xdata, *popt), 'r-')
    plt.title(f"Histogram of Article Topic Numbers for {start_date.strftime('%Y-%m')} to {end_date.strftime('%Y-%m')}")
    plt.xlabel("Topic Number")
    plt.ylabel("Frequency")
    plt.ylim(0, max(freq1[0])*1.1)
    # add annotation
    formula = f'y = {popt[0]:.4f} * exp(-{popt[1]:.4f}x) + {popt[2]:.2f}'
    plt.annotate(f'Rank: {rank}\n{formula}', xy=(len(freq)/2, max(freq)/2), xytext=(len(freq)/2, max(freq)/2), ha='center', va='center', fontsize=12, color='white', bbox=dict(facecolor='black', alpha=0.8))

    plt.show()


In [ ]:
topic_num = np.vstack([np.vstack(articles_by_month[month]['topic_num'].values) for month in articles_by_month.keys()])
triplets = [tuple(row) for row in topic_num[:, :3]]
triplet_freq = Counter(triplets)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

x, y, z, v = [], [], [], []

for key, value in triplet_freq.items():
    x.append(key[0])
    y.append(key[1])
    z.append(key[2])
    v.append(value)

v = np.array(v) / np.max(v)

cs = ax.scatter(x, y, z, c=v, cmap='jet', alpha=v, s=v*500)
#cs = ax.scatter(x[44:46], y[44:46], z[44:46], c=v[44:46], cmap='jet', alpha=1, s=v[44:46]*500)
#cs = ax.scatter(x[44], y[44], z[44], c=v[44], cmap='jet', s=v[44]*500)

ax.set_xlabel('Topic 1')
ax.set_ylabel('Topic 2')
ax.set_zlabel('Topic 3')

# add colorbar
fig.colorbar(cs)
plt.show()

#### C.2 Monthly

In [ ]:
def plot_histogram_by_month(month, rank=1, sort=True):
    topic_num = np.vstack(articles_by_month[month]['topic_num'].values)
    # determine alpha according to rank
    alpha_list = [0.9 if i==rank-1 else 0.3 for i in range(3)]
    
    freq1 = plt.hist(topic_num[:, 0], bins=range(int(topic_num[:, 0].min()), int(topic_num[:, 0].max()) + 2, 1), color='r', alpha=alpha_list[0])
    freq2 = plt.hist(topic_num[:, 1], bins=range(int(topic_num[:, 1].min()), int(topic_num[:, 1].max()) + 2, 1), color='g', alpha=alpha_list[1])
    freq3 = plt.hist(topic_num[:, 2], bins=range(int(topic_num[:, 2].min()), int(topic_num[:, 2].max()) + 2, 1), color='b', alpha=alpha_list[2])
    
    def func(x, a, b, c):
        return a*np.exp(-b*x) + c
    
    if rank==1:
        freq = freq1
    elif rank==2:
        freq = freq2
    else:
        freq = freq3
        
    if sort:
        freq = freq[0][np.argsort(freq[0])[::-1]]
    else:
        freq = freq[0]
        
    xdata = np.arange(len(freq)-1)+1
    popt, pcov = curve_fit(func, xdata, freq[1:])
    plt.bar(range(len(freq)), freq, color='k')
    plt.plot(xdata, func(xdata, *popt), 'r-')
    
     # add annotation
    formula = f'y = {popt[0]:.2f} * exp(-{popt[1]:.4f}x) + {popt[2]:.2f}'
    plt.annotate(f'Rank: {rank}\n{formula}', xy=(len(freq)/2, max(freq)/2), xytext=(len(freq)/2, max(freq)/2), ha='center', va='center', fontsize=12, color='white', bbox=dict(facecolor='black', alpha=0.8))

    plt.title(f"Histogram of Topic Numbers for {month}")
    plt.xlabel("Topic Number")
    plt.ylabel("Frequency")
    plt.show()

months = list(articles_by_month.keys())
month_widget = widgets.Dropdown(options=months, description='Month:')
rank_widget = widgets.IntSlider(min=0, max=3, step=1, value=1, description='Rank:')
sort_widget = widgets.Checkbox(value=False, description='Sort')
widgets.interactive(plot_histogram_by_month, month=month_widget, rank=rank_widget, sort=sort_widget)


In [ ]:
# for all month and each rank, get curve fitting and gather popt[1] and draw plot

sort=True
decay_list = np.zeros((3, len(articles_by_month)))
for month in list(articles_by_month.keys()):
    for rank in range(1, 4):
        topic_num = np.vstack(articles_by_month[month]['topic_num'].values)
        # determine alpha according to rank
        def func(x, a, b, c):
            return a*np.exp(-b*x) + c

        if rank==1:
            freq = np.histogram(topic_num[:, 0], bins=range(int(topic_num[:, 0].min()), int(topic_num[:, 0].max()) + 2, 1))
        elif rank==2:
            freq = np.histogram(topic_num[:, 1], bins=range(int(topic_num[:, 1].min()), int(topic_num[:, 1].max()) + 2, 1))
        else:
            freq = np.histogram(topic_num[:, 2], bins=range(int(topic_num[:, 2].min()), int(topic_num[:, 2].max()) + 2, 1))
            
        if sort:
            freq = freq[0][np.argsort(freq[0])[::-1]]
        else:
            freq = freq[0]
            
        xdata = np.arange(len(freq)-1)+1
        popt, pcov = curve_fit(func, xdata, freq[1:])
        decay_list[rank-1, list(articles_by_month.keys()).index(month)] = popt[1]

# draw decay plot for each rank
plt.figure(figsize=(10, 5))
plt.plot(list(articles_by_month.keys()), decay_list[0], label='Rank 1')
plt.plot(list(articles_by_month.keys()), decay_list[1], label='Rank 2')
plt.plot(list(articles_by_month.keys()), decay_list[2], label='Rank 3')
plt.legend()
plt.title(f"Decay of Topic Numbers for {collection_name}")
plt.xlabel("Month")
plt.ylabel("Decay")
plt.xticks(rotation=90)
plt.show()

### D. Number of Comments with topics

#### D.1. Total
* For each topic, how many comments were posted in total?

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

def plot_topic_num(sort=True, exclude_neg_one=True):
    topic_freq = np.vstack(article_topics[month] for month in article_topics.keys()).mean(axis=0)
    if exclude_neg_one:
        topic_freq = topic_freq[1:]
    if sort:
        topic_freq = sorted(topic_freq, reverse=True)
    plt.bar(range(len(topic_freq)), topic_freq, color='k')
    plt.title(f"Number of Comments per Topic for {start_date.strftime('%Y-%m')} to {end_date.strftime('%Y-%m')} ")
    plt.xlabel("Topic")
    plt.ylabel("Number of Comments")

sort_widget = widgets.Checkbox(value=True, description='Sort')
exclude_neg_one_widget = widgets.Checkbox(value=True, description='Exclude -1')

widgets.interactive(plot_topic_num, sort=sort_widget, exclude_neg_one=exclude_neg_one_widget)


In [ ]:
# find the best fitting curve for this histogram
from scipy.optimize import curve_fit

def func_exp(x, a, b, c):
    return a*np.exp(-b*x) + c

def func_quad(x, a, b, c):
    return 1 / (a * x ** 2 + b * x + c)

def plot_topic_freq(func_type='exp', exclude_neg_one=True):

    topic_freq = np.vstack(article_topics[month] for month in article_topics.keys()).mean(axis=0)
    if exclude_neg_one:
        topic_freq = topic_freq[1:]

    topic_freq = sorted(topic_freq, reverse=True)
    xdata = np.arange(len(topic_freq))
    
    if func_type == 'exp':
        popt, pcov = curve_fit(func_exp, xdata, topic_freq)
        func = func_exp
        formula = f'y = {popt[0]:.2f} * {func_type}(-{popt[1]:.2f}x) + {popt[2]:.2f}'
    elif func_type == 'quad':
        popt, pcov = curve_fit(func_quad, xdata, topic_freq)
        func = func_quad
        formula = f'y = 1 / ({popt[0]:.2f} * x^2 + ({popt[1]:.2f}) * x + {popt[2]:.2f})'
    else:
        raise ValueError('Invalid function type. Choose between "exp" and "power"')
    
    plt.bar(range(len(topic_freq)), topic_freq)
    plt.plot(xdata, func(xdata, *popt), 'r-')
    plt.title(f"Histogram of Comment Topic Numbers")
    plt.xlabel("Topic Number")
    plt.ylabel("Frequency")
    
    plt.annotate(f'{formula}', xy=(len(topic_freq)/2, max(topic_freq)/2), xytext=(len(topic_freq)/2, max(topic_freq)/2), ha='center', va='center', fontsize=12, color='white', bbox=dict(facecolor='black', alpha=0.8))
    plt.show()


exclude_neg_one_widget = widgets.Checkbox(value=True, description='Exclude -1')
widgets.interactive(plot_topic_freq, func_type=['exp', 'quad'], exclude_neg_one=exclude_neg_one_widget)


#### D.2. Monthly
* For each topic, how many comments were posted each month?

In [ ]:
def plot_topic_num_by_month(month, sort=True, exclude_neg_one=True):
    topic_freq = article_topics[month]
    if exclude_neg_one:
        topic_freq = topic_freq[1:]
    if sort:
        topic_freq = sorted(topic_freq, reverse=True)
    plt.bar(range(len(topic_freq)), topic_freq, color='k')
    plt.title(f"Number of Comments per Topic for {month}")
    plt.xlabel("Topic")
    plt.ylabel("Number of Comments")
    plt.show()

months = list(articles_by_month.keys())
month_widget = widgets.Dropdown(options=months, description='Month:')
sort_widget = widgets.Checkbox(value=True, description='Sort')
exclude_neg_one_widget = widgets.Checkbox(value=True, description='Exclude -1')
widgets.interactive(plot_topic_num_by_month, month=month_widget, sort=sort_widget, exclude_neg=exclude_neg_one_widget)


#### D.3. Monthly Top rankings
* For each month, which topics were the at the top (20)?

In [ ]:
# with plot_month method, make a widget that shows the frequency of each topic sorted by 'rank' for each month.

# take month as 'month' and plot the frequency of each topic sorted by 'rank' for that month. show only top N topics.
def plot_month(df, month, exclude_neg_one=True, N=20):
    
    topic_freq = article_topics[month]
    
    if exclude_neg_one:
        topic_freq = topic_freq[1:]
        df = pd.DataFrame({'Topic': range(len(topic_freq)), 'Frequency': topic_freq})
    else:
        df = pd.DataFrame({'Topic': range(-1, len(topic_freq)-1), 'Frequency': topic_freq})
    
    # sort the dataframe by frequency and get the top 20 topics
    df = df.sort_values(by='Frequency', ascending=False).head(N)
    # plot the top 20 topics in a bar graph
    df.plot(x='Topic', y='Frequency', kind='bar')
    
    plt.xlabel('Topic', fontsize=14)
    plt.ylabel('normalized frequency', fontsize=14)
    
    # Get the text of current x tick labels and makes it as a list
    labels = [item.get_text() for item in plt.gca().get_xticklabels()]
    if exclude_neg_one:
        labels_int = np.array([int(x) for x in labels])
        keywords = [[x.split('_')[1] for x in topic_model.get_topic_info()['Name']][1:][j] for j in labels_int]
    else:
        labels_int = np.array([int(x) for x in labels])+1
        keywords = [[x.split('_')[1] for x in topic_model.get_topic_info()['Name']][j] for j in labels_int]

    # Adding keywords at the end of the x tick labels
    labels = [f'{labels[i]}, {keywords[i]}' for i in range(N)]
    plt.xticks(range(N), labels, rotation=45, ha='right')                

months = list(articles_by_month.keys())
exclude_neg_one_widget = widgets.Checkbox(value=True, description='Exclude -1')
@interact(month=months, exclude_neg_one=exclude_neg_one_widget)
def plot_month_widget(month=months, exclude_neg_one=True, N=(3, 50)):
    plot_month(df2, month, exclude_neg_one, N)


#### D.4. Normalized frequency / Ranking of comment topics
* For each topic, what is the normalized frequency / ranking for each month?

In [ ]:
# take topic as 'topic' and plot the frequency of each topic for all months in the df.
topic_freqs = np.stack(topic_by_month.values())
topic_freqs_eno = topic_freqs[:, 1:]

topic_freqs = topic_freqs / topic_freqs.sum(axis=1)[..., np.newaxis]
topic_freqs_eno = topic_freqs_eno / topic_freqs_eno.sum(axis=1)[..., np.newaxis]

topic_ranking = np.argsort(topic_freqs, axis=1)[:, ::-1]
topic_ranking_eno = np.argsort(topic_freqs_eno, axis=1)[:, ::-1]

def plot_topic(topic, ytype, exclude_neg_one=True):
    fig = plt.figure(figsize=(10, 7))
    if exclude_neg_one:
        if topic == -1:
            print('nothing to plot')
        else:
            if ytype == 'norm_freq':
                plt.bar(articles_by_month.keys(), topic_freqs_eno[:, topic], color='k')
            elif ytype == 'rank':
                plt.bar(articles_by_month.keys(), topic_ranking_eno[:, topic], color='k')
            else:
                print('error')
    else:
        if ytype == 'norm_freq':
            plt.bar(articles_by_month.keys(), topic_freqs[:, topic+1], color='k')
        elif ytype == 'rank':
            plt.bar(articles_by_month.keys(), topic_ranking[:, topic+1], color='k')
        else:
            print('error')
        
    plt.xticks(rotation=90)
    plt.xlabel('Month', fontsize=14)
    plt.ylabel(ytype, fontsize=14)
    
# with plot_month method, make a widget that shows the frequency of each topic sorted by 'rank' for each month.

topics = np.arange(-1, num_topics)
@interact
def plot_topic_widget(topic=topics, ytype=['norm_freq', 'rank'], exclude_neg_one=True):
    plot_topic(topic, ytype, exclude_neg_one)

## 2. Title topic distribution
* What is the distribution of top 3 title topics?

### A. Total

In [ ]:
@interact(rank=widgets.IntSlider(min=0, max=3, step=1, value=1), normalize=False)
def plot_histogram_by_month(rank, normalize=False):
    topic_num = np.vstack([np.vstack(articles_by_month[month]['topic_num'].values) for month in articles_by_month.keys()])
    alpha_list = [1 if i==rank-1 else 0.3 for i in range(3)]

    freq1 = plt.hist(topic_num[:, 0], bins=range(int(topic_num[:, 0].min()), int(topic_num[:, 0].max()) + 2, 1), color='r', alpha=alpha_list[0], density=normalize)
    freq2 = plt.hist(topic_num[:, 1], bins=range(int(topic_num[:, 1].min()), int(topic_num[:, 1].max()) + 2, 1), color='g', alpha=alpha_list[1], density=normalize)
    freq3 = plt.hist(topic_num[:, 2], bins=range(int(topic_num[:, 2].min()), int(topic_num[:, 2].max()) + 2, 1), color='b', alpha=alpha_list[2], density=normalize)
    plt.title(f"Histogram of Article Topic Numbers for {start_date.strftime('%Y-%m')} to {end_date.strftime('%Y-%m')}")
    plt.xlabel("Topic Number")
    plt.ylabel("Frequency")    
    plt.show()

### B. Topic

In [ ]:
def plot_by_topic(topic_num, normalize=True):
    plt.figure(figsize=(10, 5))
    topic_num_list = [[] for _ in range(3)]
    color_list = ['r', 'g', 'b']
    
    for month, articles in articles_by_month.items():
        if normalize:
            constant = len(articles_by_month[month])
        else:
            constant = 1
        for i in range(3):
            topic_num_list[i].append(len(articles[np.vstack(articles['topic_num'])[:, i] == topic_num])/constant)
    
    for i in range(2, -1, -1):
        plt.bar(list(articles_by_month.keys()), topic_num_list[i], color=color_list[i], alpha=0.5)
        
    plt.title(f"Plot of article frequency for topic {topic_num}")
    plt.xlabel("Month")
    plt.ylabel("Frequency")
    plt.xticks(rotation=90)
    plt.show()

topic_num_widget = widgets.IntSlider(min=0, max=num_topics, step=1, value=0, description='Topic Number:')
normalize_widget = widgets.Checkbox(value=True, description='Normalize')
widgets.interactive(plot_by_topic, topic_num=topic_num_widget, normalize=normalize_widget)

# Atlantic, topic 51 is super-seasonal (3~5)

### C. Month / Article

In [ ]:
def plot_topic_distribution_by_month(month, topic_num, article_id, exclude_neg_one=False):
    articles = articles_by_month[month]
    articles = articles[np.vstack(articles['topic_num'])[:, 0] == topic_num]
    article_ids = articles['_id'].values.tolist()
    if len(article_ids) > 0:
        article = articles[articles['_id'] == article_id]
        if len(article) > 0:
            comment_topics = article['comment_topics'].values[0]
            if exclude_neg_one:
                comment_topics = [t for t in comment_topics if t != -1]
            freq = plt.hist(comment_topics, bins=range(int(min(comment_topics)), int(max(comment_topics)) + 2, 1))
            plt.axvline(x=article['topic_num'].values[0][0], linestyle='--', color='red', alpha=0.5)

            plt.title(f"Histogram of Topic Numbers for Article {article_id} (Topic {article['topic_num'].values[0][0]})")
            plt.xlabel("Topic Number")
            plt.ylabel("Frequency")
            plt.xticks(np.arange(min(comment_topics), max(comment_topics)+1, 10))
            plt.xticks(rotation=90)
            plt.yticks(np.arange(0, max(freq[0])+1, 1 if max(freq[0]) < 10 else 5))

            plt.show()
        else:
            print(f"No article found with ID {article_id} and topic number {topic_num}")
    else:
        print(f"No articles found with topic number {topic_num} in month {month}")

months = list(articles_by_month.keys())
month_widget = widgets.Dropdown(options=months, description='Month:')

articles = articles_by_month[months[0]]
articles = articles[np.vstack(articles['topic_num'])[:, 0] == topic_num_widget.value]
topic_nums = np.unique(np.vstack(articles['topic_num']).flatten())
topic_num_widget = widgets.IntSlider(min=topic_nums.min(), max=topic_nums.max(), step=1, value=topic_nums.min(), description='Topic Number:')

article_ids = articles[articles['comment_id'].apply(len) > threshold]['_id'].values.tolist()
article_id_widget = widgets.Dropdown(options=article_ids, description='Article ID:')

def update_topic_nums(*args):
    articles = articles_by_month[month_widget.value]
    topic_nums = np.unique(np.vstack(articles['topic_num']).flatten())
    topic_num_widget.min = topic_nums.min()
    topic_num_widget.max = topic_nums.max()
    topic_num_widget.value = topic_nums.min()

month_widget.observe(update_topic_nums, 'value')

def update_article_ids(*args):
    articles = articles_by_month[month_widget.value]
    articles = articles[np.vstack(articles['topic_num'])[:, 0] == topic_num_widget.value]
    article_ids = articles[articles['comment_id'].apply(len) > 1]['_id'].values.tolist()
    article_id_widget.options = article_ids

topic_num_widget.observe(update_article_ids, 'value')
widgets.interactive(plot_topic_distribution_by_month, month=month_widget, topic_num=topic_num_widget, article_id=article_id_widget)


### D. Seasonality



In [ ]:
def plot_signal_and_fft(topic):
    signal = norm_topic_num_list[:, topic, 0]
    fft = np.fft.fftshift(signal)
    freqs = np.fft.fftfreq(len(signal))
    freqs = np.concatenate((freqs[len(freqs)//2+1:], freqs[:len(freqs)//2+1]))    
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(signal)
    ax1.set_title('Signal')
    
    ax2.plot(freqs, np.abs(fft))
    ax2.axvline(x=1/6, color='r', linestyle='--', alpha=0.5)
    ax2.axvline(x=1/12, color='r', linestyle='--')
    ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax2.axvline(x=-1/6, color='r', linestyle='--', alpha=0.5)
    ax2.axvline(x=-1/12, color='r', linestyle='--')
    ax2.set_title('FFT')
    
    plt.show()

topic_slider = widgets.IntSlider(min=0, max=norm_topic_num_list.shape[1]-1, step=1, value=0)
widgets.interactive(plot_signal_and_fft, topic=topic_slider)
# atlantic, topic 51


### E. Autocorrelation



In [ ]:
def autocorrelation(x):
        """
        Calculate the autocorrelation of a given 1D array x.
        """
        n = len(x)
        variance = x.var()
        x = x - x.mean()
        r = np.correlate(x, x, mode='full')[-n:]
        result = r / (variance * (np.arange(n, 0, -1)))
        return result

def plot_autocorrelation(rank):
    data = norm_topic_num_list[:, :, rank-1].T
    # calculate autocorrelation of each row
    autocorr = np.array([autocorrelation(data[i]) for i in range(len(data))])
    plt.plot(autocorr[:100].mean(axis=0))
    plt.xlabel('Lag')
    plt.ylabel('Autocorrelation')
    plt.title(f'Autocorrelation for rank {rank}')
    plt.show()

rank_slider = widgets.IntSlider(min=1, max=3, step=1, value=1, description='Rank:')


widgets.interactive(plot_autocorrelation, rank=rank_slider)


In [ ]:
# check whether in df, column ['article_id'] has a duplicate.
df[df.duplicated(['article_id'])]

In [ ]:
df.set_index('Month', inplace=True)
df

## 3. Comments topic graph

### A. Comments topic embedding space
* For each month, how mean topic embeddings are placed in (2-dim) embedding space?

In [ ]:
from scipy.spatial.distance import pdist

distances_list = []
for i in range(num_topics+1):
    distances_list.append(np.array([np.linalg.norm(total_mean_embedding_list[j, i, :] - total_mean_embedding_list[j+1, i, :]) for j in range(len(total_mean_embedding_list)-1)]))

distances_list = np.array(distances_list)

In [ ]:
distances_list = []
for i in range(num_topics+1):
    distances_list.append(np.array([np.linalg.norm(total_tsne_list[j, i, :] - total_tsne_list[j+1, i, :]) for j in range(len(total_tsne_list)-1)]))

distances_list = np.array(distances_list)

In [ ]:
plt.scatter(np.arange(len(distances_list)), distances_list.mean(axis=1))

In [ ]:
plt.scatter(np.arange(len(distances_list)), distances_list.mean(axis=1))

In [ ]:
# visualizing total_tsne_list, which has 92 images of topic_num points with 2d coordinates, with widgets.
# add toggle button to show the movement of points
# set default i to 0


@interact
def show_tsne(i=(0, len(articles_by_month.keys())-1), focal=(-1, num_topics), movement=True, total=True,  exclude_neg_one=True):
    
    fig = plt.figure(figsize=(5, 4), dpi=200)
    ax = fig.add_subplot(111)
    
    if exclude_neg_one:
        
        jet_colors = plt.get_cmap('jet')(np.linspace(0, 1, num_topics))
        total_tsne_list
        # Assign every point a different color on the jet colormap
        if total: 
            im = ax.scatter(total_tsne_list[i][1:, 0], total_tsne_list[i][1:, 1], c=np.arange(num_topics), cmap='jet', s=5)
            # if i!=0, plot the previous points with alpha=0.2 to show the movement of points
            if movement and i!=0:
                ax.scatter(total_tsne_list[i-1][1:, 0], total_tsne_list[i-1][1:, 1], c=np.arange(num_topics), cmap='jet', s=5, alpha=0.2)
                # draw arrows to show the movement of points with alpha 0.2, with arrowhead length 0.1
                for j in range(num_topics):
                    ax.arrow(total_tsne_list[i-1][j+1, 0], total_tsne_list[i-1][j+1, 1], total_tsne_list[i][j+1, 0]-total_tsne_list[i-1][j+1, 0], total_tsne_list[i][j+1, 1]-total_tsne_list[i-1][j+1, 1], color=jet_colors[j], head_width=1.5, alpha=0.2)
        else:
            if focal==-1:
                print('nothing to plot')
            else:
                # scatter a single point (focal) with color of focal point with jet colormap
                im = ax.scatter(total_tsne_list[i][1:, 0], total_tsne_list[i][1:, 1], c=np.arange(num_topics), cmap='jet', s=5, alpha=0.2)
                im2 = ax.scatter(total_tsne_list[i][focal+1, 0], total_tsne_list[i][focal+1, 1], c=jet_colors[focal], s=10, alpha=1)
                # if i!=0, plot the previous points with alpha=0.2 to show the movement of points
                if movement and i!=0:
                    ax.scatter(total_tsne_list[i-1][focal+1, 0], total_tsne_list[i-1][focal+1, 1], c=jet_colors[focal], s=10, alpha=0.5)
                    # draw arrows to show the movement of points with alpha 0.2, with arrowhead length 0.1
                    ax.arrow(total_tsne_list[i-1][focal+1, 0], total_tsne_list[i-1][focal+1, 1], total_tsne_list[i][focal+1, 0]-total_tsne_list[i-1][focal+1, 0], total_tsne_list[i][focal+1, 1]-total_tsne_list[i-1][focal+1, 1], head_width=2.5, alpha=1)
    else:
        jet_colors = plt.get_cmap('jet')(np.linspace(0, 1, num_topics+1))
    
        # Assign every point a different color on the jet colormap
        if total: 
            im = ax.scatter(total_tsne_list[i][:, 0], total_tsne_list[i][:, 1], c=np.arange(num_topics+1), cmap='jet', s=5)
            # if i!=0, plot the previous points with alpha=0.2 to show the movement of points
            if movement and i!=0:
                ax.scatter(total_tsne_list[i-1][:, 0], total_tsne_list[i-1][:, 1], c=np.arange(num_topics+1), cmap='jet', s=5, alpha=0.2)
                # draw arrows to show the movement of points with alpha 0.2, with arrowhead length 0.1
                for j in range(-1, num_topics):
                    ax.arrow(total_tsne_list[i-1][j+1, 0], total_tsne_list[i-1][j+1, 1], total_tsne_list[i][j+1, 0]-total_tsne_list[i-1][j+1, 0], total_tsne_list[i][j+1, 1]-total_tsne_list[i-1][j+1, 1], color=jet_colors[j+1], head_width=1.5, alpha=0.2)
        else:
            # scatter a single point (focal) with color of focal point with jet colormap
        
            im = ax.scatter(total_tsne_list[i][:, 0], total_tsne_list[i][:, 1], c=np.arange(num_topics+1), cmap='jet', s=5, alpha=0.2)
            im2 = ax.scatter(total_tsne_list[i][focal+1, 0], total_tsne_list[i][focal+1, 1], c=jet_colors[focal+1], s=10, alpha=1)
            # if i!=0, plot the previous points with alpha=0.2 to show the movement of points
            if movement and i!=0:
                ax.scatter(total_tsne_list[i-1][focal+1, 0], total_tsne_list[i-1][focal+1, 1], c=jet_colors[focal+1], s=10, alpha=0.5)
                # draw arrows to show the movement of points with alpha 0.2, with arrowhead length 0.1
                ax.arrow(total_tsne_list[i-1][focal+1, 0], total_tsne_list[i-1][focal+1, 1], total_tsne_list[i][focal+1, 0]-total_tsne_list[i-1][focal+1, 0], total_tsne_list[i][focal+1, 1]-total_tsne_list[i-1][focal+1, 1], head_width=2.5, alpha=1)
    
    ax.set_xlim(-100, 100)
    ax.set_ylim(-100, 100)
    # Add a colorbar to show the meaning of the colors
    ax.set_xlabel('TSNE 1')
    ax.set_ylabel('TSNE 2')
    ax.set_title(f'TSNE of topics in {list(articles_by_month.keys())[i] + "/" + list(articles_by_month.keys())[i+1]}')
    
    # add a colorbar name
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Topic ID')

### B. Comments topic similarity
* What is the distribution and tendency of the similarity between topics?

In [ ]:
# make a widget that draws distribution of distance_matrix of each month
@interact
def plot_distance_matrix(i=(0, len(articles_by_month.keys())-1)):
    plt.figure(figsize=(10, 7))
    # flatten the distance matrix and exclude the diagonal term
    flattened = distance_matrix_list[i].flatten()
    flattened = flattened[flattened != 1]
    plt.hist(flattened, bins=20)
    plt.xlabel('Cosine Similarity')
    plt.ylabel('Frequency')
    plt.title(f'Distribution of Cosine Similarity in {list(articles_by_month.keys())[i]}')
    
    plt.show()

In [ ]:
# make a widget that draws histogram of mean_weight_list[i] for each i
@interact
def plot_mean_weight(i=(0, len(articles_by_month.keys())-1)):
    plt.figure(figsize=(10, 7))
    x = distance_matrix_list[i]
    plt.hist(np.ravel(x[1:, 1:])[~np.eye(x.shape[0]-1, dtype=bool).flatten()], bins=20)
    plt.xlabel('Embedding distance')
    plt.ylabel('Frequency')
    plt.title(f'Mean Weight of {num_topics} topics in {list(articles_by_month.keys())[i]}')
    plt.show()

In [ ]:
# draw a plot of variance of mean_weight_list
fig, axs = plt.subplots(1, 2, figsize=(16, 7))

axs[0].plot(np.arange(len(distance_matrix_list)), [np.mean(np.ravel(x[1:, 1:])[~np.eye(x.shape[0]-1, dtype=bool).flatten()]) for x in distance_matrix_list])
axs[0].set_title('Mean of mean_weight_list')

axs[1].plot(np.arange(len(distance_matrix_list)), [np.var(np.ravel(x[1:, 1:])[~np.eye(x.shape[0]-1, dtype=bool).flatten()]) for x in distance_matrix_list])
axs[1].set_title('Variance of mean_weight_list')

plt.show()


In [ ]:
# make a widget that draws histogram of mean_weight_list[i] for each i
@interact
def plot_mean_weight(i=(0, len(articles_by_month.keys())-1)):
    plt.figure(figsize=(10, 7))
    plt.hist(mean_weight_list[i], bins=20)
    plt.xlabel('Mean Weight')
    plt.ylabel('Frequency')
    plt.title(f'Mean Weight of {num_topics} topics in {list(articles_by_month.keys())[i]}')
    plt.show()

### C. Correlaton between frequency and similarity of comment topics
* Does topics with higher frequency are more similar to other topics?

In [ ]:
order_index_list = []
# sort the distance matrix by the frequency of each topic at each month
distance_matrix_sorted_list = []
for i in range(len(distance_matrix_list)):
    freq = deepcopy(topic_by_month[list(articles_by_month.keys())[i]])
    for index in sorted(total_nonexist_index_list[i], reverse=True):
        freq.pop(index)
    order_index = np.argsort(freq)[::-1]
    distance_matrix_sorted = distance_matrix_list[i][order_index]
    distance_matrix_sorted = distance_matrix_sorted[:, order_index]
    order_index_list.append(order_index)
    distance_matrix_sorted_list.append(distance_matrix_sorted)

@interact
def plot_distance_matrix(i=(0, len(articles_by_month.keys())), ordered=True):
    if ordered:
        plt.imshow(distance_matrix_sorted_list[i], cmap='hot')
    else:
        plt.imshow(distance_matrix_list[i], cmap='hot')
    # colorbar
    plt.colorbar()
    plt.title(f"Distance Matrix for {list(articles_by_month.keys())[i]}")
    plt.show()
    

### D. Change of pairwise similairty between topics
* For topic pair (i, j), how its similarity changed by time?

In [ ]:
def get_pair_sim(i, j):
    sim_list = []
    sim_nonexist_list = []

    for k in range(len(distance_matrix_list)):
        real_i, real_j = i+1, j+1
        if real_i in total_nonexist_index_list[k] or real_j in total_nonexist_index_list[k]:
            sim_nonexist_list.append(-1)
            if k==0:
                sim_list.append(0)
            else:
                sim_list.append(sim_list[-1])
        else:
            sim_nonexist_list.append(0)
            for x in range(-1, i+1):
                if x in total_nonexist_index_list[k]:
                    real_i -= 1
            
            for y in range(-1, j+1):
                if y in total_nonexist_index_list[k]:
                    real_j -= 1

            sim_list.append(distance_matrix_list[k][real_i, real_j])

    return sim_list, sim_nonexist_list

@interact
def plot_pair_sim(i=widgets.IntSlider(min=0, max=num_topics-1, step=1, value=0),
                    j=widgets.IntSlider(min=0, max=num_topics-1, step=1, value=1)):
    sim_list, sim_nonexist_list = get_pair_sim(i, j)
    plt.figure(figsize=(10, 7))
    plt.plot(sim_list)
    plt.xlabel('Month')
    plt.ylabel('Similarity')
    plt.title(f'Similarity between topic {i} and topic {j}')
    plt.show()

## 4. Title (T) - Comments (T) relation

### A. Article - Comments gap
* After the article is posted, how long after the comments were posted?

In [ ]:
comments_createdAt_all = Counter()
for month, articles in articles_by_month.items():
    for _, article in articles.iterrows():
        comments_createdAt_all += Counter([(cc - article['createdAt']).days for cc in article['comment_createdAt']])

sorted_counter = sorted(comments_createdAt_all.items())
x, y = zip(*sorted_counter)

# exclude -1 (possibly due to some errors in the DB, ~ 0.01% of comments are recorded before the article is created)
x = x[1:]
y = y[1:]

# normalize y
y_norm = [i/sum(y) for i in y]

# calculate cumulative sum of y
y_cumsum = np.cumsum(y_norm)

plt.bar(x, y_cumsum)
percentage_7 = y_cumsum[7-1] * 100
percentage_15 = y_cumsum[15-1] * 100
percentage_30 = y_cumsum[30-1] * 100

print(f"{percentage_7:.2f}% of cumulative comments were made within 7 days.")
print(f"{percentage_15:.2f}% of cumulative comments were made within 15 days.")
print(f"{percentage_30:.2f}% of cumulative comments were made within 30 days.")
plt.title('Histogram of comments_createdAt')
plt.xlabel('Days after article creation')
plt.ylabel('Cumulative Frequency')
plt.show()

### B. Title topics vs. Comments topics
* For top 3 title topics, does article has more comments with those topics than others?

In [ ]:
comment_topics_by_topic = {}
for topic_num in range(num_topics+1):  # including -1
    comment_topics = [[] for _ in range(3)]
    for month, articles in articles_by_month.items():
        for i in range(3):
            articles_tmp = articles[np.vstack(articles['topic_num'])[:, i] == topic_num]
            article_ids = articles_tmp['_id'].values.tolist()
            if len(article_ids) > 0:
                for article_id in article_ids:
                    article_tmp = articles_tmp[articles_tmp['_id'] == article_id]
                    if len(article) > 0:
                        comment_topics[i].extend(article_tmp['comment_topics'].values[0])
    comment_topics_by_topic[topic_num] = comment_topics


In [ ]:
def plot_topic_distribution(topic_num, normalized=False, exclude_neg_one=False):
    comment_topics = comment_topics_by_topic[topic_num]
    color_list = ['r', 'g', 'b']
    if exclude_neg_one:
        for i in range(3):
            freq, bins, _ = plt.hist([t for t in comment_topics[i] if t != -1], bins=range(0, num_topics-1, 1), density=normalized, color=color_list[i], alpha=0.3)
    else:
        for i in range(3):
            freq, bins, _ = plt.hist(comment_topics[i], bins=range(-1, num_topics-1, 1), density=normalized, color=color_list[i], alpha=0.3)
    plt.axvline(x=topic_num, linestyle='--', color='red', alpha=0.5)
    plt.title(f"Histogram of Comments Topic Numbers for Topic {topic_num}")
    plt.xlabel("Topic Number")
    if normalized:
        plt.ylabel("Normalized Frequency")
    else:
        plt.ylabel("Frequency")
    plt.xticks(np.arange(0, num_topics, 10))
    plt.xticks(rotation=90)
    plt.show()
    
topic_num_widget = widgets.IntSlider(min=0, max=num_topics-1, step=1, value=0, description='Topic Number:')
widgets.interactive(plot_topic_distribution, topic_num=topic_num_widget)

In [ ]:
# draw 3d plot with histograms of comment_topics_by_topic for each topic number, let x is topic number of article, y is topic number of comment, z is frequency. Not a widget, a single 3d plot, multiple 2d histograms in 3d plot.

fig = plt.figure(figsize=(20, 12))
ax = fig.add_subplot(111, projection='3d')
exclude_neg_one = True
rank = 2

self_freq = []
for topic_num in range(num_topics):
    
    comment_topics = comment_topics_by_topic[topic_num][rank-1]
    if exclude_neg_one:
        x = np.arange(0, num_topics, 1)
        comment_topics = [t for t in comment_topics if t != -1]
        freq, bins = np.histogram(comment_topics, bins=range(0, num_topics+1, 1))
        freq_norm = freq/np.sum(freq)
        ax.bar3d(x, np.ones_like(x) * topic_num, 0, 1, 1, freq_norm, color='b', alpha=0.1)
    else:
        x = np.arange(-1, num_topics, 1)
        freq, bins = np.histogram(comment_topics, bins=range(-1, num_topics, 1))
        freq_norm = freq/np.sum(freq)
        ax.bar3d(x, np.ones_like(x) * topic_num, 0, 1, 1, freq_norm, color='b', alpha=0.1)
   
    self_freq.append(freq_norm[topic_num])
    
ax.bar3d(x, x, 0, 1, 1, self_freq, color='red', alpha=0.3)
ax.plot([0, num_topics-1], [0, num_topics-1], linestyle='--', color='red', alpha=0.5)
ax.set_xlabel('Comment Topic Number')
ax.set_ylabel('Article Topic Number')
ax.set_zlabel('Frequency')
ax.set_title(f"3D Histogram of Topic Numbers")
plt.show()


In [ ]:
# for each month, calculate the total frequency of comment's topic number
def get_topic_frequency_by_month(articles_by_month, exclude_neg_one=False):
    topic_freq_by_month = {}
    for month, articles in articles_by_month.items():
        comment_topics = []
        for i in range(len(articles)):
            article = articles.iloc[i]
            comment_topics.extend(article['comment_topics'])
        if exclude_neg_one:
            comment_topics = [t for t in comment_topics if t != -1]
        topic_freq_by_month[month] = Counter(comment_topics)
    return topic_freq_by_month

# sum all counters for all months
def get_topic_frequency(topic_freq_by_month):
    topic_freq = Counter()
    for month, freq in topic_freq_by_month.items():
        topic_freq += freq
    return topic_freq

def get_topic_frequency_by_month_self(articles_by_month, rank, exclude_neg_one=False):
    topic_freq_by_month = {}
    for month, articles in articles_by_month.items():
        comment_topics = defaultdict(list)
        for i in range(len(articles)):
            article = articles.iloc[i]
            article_topic = article['topic_num'][rank-1]
            if exclude_neg_one:
                # if article topic is not -1
                if article_topic != -1:
                    comment_topics[article_topic].extend(article['comment_topics'])
        if exclude_neg_one:
            for key in comment_topics.keys():
                comment_topics[key] = [t for t in comment_topics[key] if t != -1]
        topic_freq_by_month[month] = {}
        for key in comment_topics.keys(): 
            topic_freq_by_month[month][key] = Counter(comment_topics[key])
    return topic_freq_by_month

# sum all counter for all topics for all months
def get_topic_frequency_self(topic_freq_by_month_self):
    topic_freq = defaultdict(Counter)
    for month, freq in topic_freq_by_month_self.items():
        for topic, f in freq.items():
            topic_freq[topic]+=f
    return topic_freq


In [ ]:
@interact(rank=(1, 5), exclude_neg_one=True)
def plot_self_ratio(rank, exclude_neg_one):
    topic_freq_by_month = get_topic_frequency_by_month(articles_by_month, exclude_neg_one)
    topic_freq = get_topic_frequency(topic_freq_by_month)
    topic_freq_by_month_self = get_topic_frequency_by_month_self(articles_by_month, rank, exclude_neg_one)
    topic_freq_self = get_topic_frequency_self(topic_freq_by_month_self)

    ratio_list = []
    for target, f in topic_freq_self.items():
        self_ratio = topic_freq_self[target][target] / topic_freq_self[target].total()
        overall_ratio = (topic_freq[target]-topic_freq_self[target][target]) / (topic_freq.total()-topic_freq_self[target].total())
        ratio_list.append(self_ratio/overall_ratio)
        
    # make integer bins
    bins = np.arange(0, 25, 0.25)
    fig, axs = plt.subplots(ncols=2, figsize=(15,5))

    result = axs[0].hist(ratio_list, bins=bins, density=True, cumulative=True)
    over_one_ratio = result[0][3]
    axs[0].axvline(x=1.0, linestyle='--', color='red', alpha=0.5)
    axs[0].axhline(y=over_one_ratio, linestyle='--', color='red', alpha=0.5)
    axs[0].text(5, 0.5, f"Topic-wise, {(1-over_one_ratio)*100:.1f}% of topics have more comments \n of its own topic than expected ")
    axs[0].set_xlabel('Self / Others Ratio')
    axs[0].set_ylabel('Cumulative density')

    topic_num_articles = defaultdict(int)
    for month, articles in articles_by_month.items():
        for topic in articles['topic_num'].values:
            topic_num_articles[topic[rank-1]] += 1
    
    if exclude_neg_one:
        result = axs[1].hist(ratio_list, bins=bins, density=True, cumulative=True, weights=list(dict(sorted(topic_num_articles.items(), key=lambda item: item[0])).values())[1:])
    else:
        result = axs[1].hist(ratio_list, bins=bins, density=True, cumulative=True, weights=list(dict(sorted(topic_num_articles.items(), key=lambda item: item[0])).values()))
    over_one_ratio = result[0][3]
    axs[1].axvline(x=1.0, linestyle='--', color='red', alpha=0.5)
    axs[1].axhline(y=over_one_ratio, linestyle='--', color='red', alpha=0.5)
    axs[1].text(5, 0.5, f"Article-wise, {(1-over_one_ratio)*100:.1f}% of articles have more comments \n of its own topic than expected.")
    axs[1].set_xlabel('Self / Others Ratio')
    axs[1].set_ylabel('Cumulative density')

plt.show()

# AT, (70.1, 71.0), (57.8, 65.9), (53.9, 56.7), (57.1, 56.8), (54.8, 56.3)
# MJ, (83.3, 91.8), (87.7, 93.5), (87.4, 92.3), (84.2, 90.3), (83.2, 82.2)
 

### C. Title topics vs. Comments topics pair-wise similarity
* For top 3 title topics, does the cosine similarity between the comments of those topics are greater than others?

In [ ]:
# load ratio_dict
with open(join('article', collection_name.lower(), model_name, 'ratio_dict.pkl'), 'rb') as f:
    ratio_dict = pickle.load(f)
    
average_ratio = np.mean(list(ratio_dict.values()), axis=0)
print(average_ratio)

In [ ]:
# extract key triplets and value lists from ratio_dict
key_triplets = []
value_lists = []
for key, value in ratio_dict.items():
    key_triplets.append(key)
    value_lists.append(value)

# calculate sum of key triplets and mean of value lists
x = [sum(key) for key in key_triplets]
y = [sum(values)/len(values) for values in value_lists]

# plot x and y
plt.scatter(x, y)
plt.xlabel('Sum of key triplet')
plt.ylabel('Mean of value list')
plt.show()


In [ ]:
# load csv file into pandas dataframe
df = pd.read_csv(join('article', collection_name.lower(), model_name, 'ratio_dict.csv'))

## 5. Comments (T) vs. Title (T+1) relation

In [ ]:
# Comments frequency (T)

comments_frequency = np.array(list(topic_by_month.values()))[:, 1:]  # removing -1
row_sums = comments_frequency.sum(axis=1)
comments_frequency = comments_frequency / row_sums[:, np.newaxis]
comments_frequency_diff = np.diff(comments_frequency, axis=0)

In [ ]:
# Comments similarity (T)

distance_matrix_full_list = []
for i in range(len(total_mean_embedding_list)):
    x = total_mean_embedding_list[i]
    # for x, I want to get a distance matrix between all pairs
    distance_matrix = get_distance_matrix(x)
    distance_matrix_full_list.append(distance_matrix[1:, 1:]) # removing -1
    
comments_similarity = np.array(distance_matrix_full_list)

In [ ]:
# Title frequency (T+1 - T)

diff_title_1 = np.diff(norm_topic_num_list[:, :, 0], axis=0)
diff_title_2 = np.diff(norm_topic_num_list[:, :, 1], axis=0)
diff_title_3 = np.diff(norm_topic_num_list[:, :, 2], axis=0)

In [ ]:
# Title pair frequency (T+1 - T)

topic_pair_list_dict = defaultdict(Counter)
for month in articles_by_month.keys():
    articles = articles_by_month[month]
    for article in articles.itertuples():
        if -1 not in article.topic_num[:3]:
            topic_pair_list_dict[month].update(list(combinations(tuple(article.topic_num[:3]), 2)))

In [ ]:
diff_pair_freq = []
for i, month in enumerate(articles_by_month.keys()):
    if i:
        copy = topic_pair_list_dict[month].copy()
        copy.subtract(topic_pair_list_dict[prev_month])
        diff_pair_freq.append(copy)
    prev_month = month

### A. Frequency

In [ ]:
def moving_average(values, window_size):
    moving_averages = []
    for i in range(len(values) - window_size + 1):
        window = values[i:i+window_size]
        window_average = sum(window) / window_size
        moving_averages.append(window_average)
    return np.array(moving_averages)

In [ ]:
corr_list = np.zeros((24, 24))
for window_size in range(1, 24):
    for shift_size in range(24):
        ma = moving_average(comments_frequency_diff, window_size)
        freq_corr = np.corrcoef(ma[:-(1+shift_size)].flatten(), diff_title_1[window_size+shift_size:].flatten())
        corr_list[window_size-1][shift_size]= freq_corr[0, 1]
    
plt.imshow(corr_list)
plt.colorbar()
plt.xlabel('Shift size')
plt.ylabel('Window size')


In [ ]:
shift_size = 0
window_size = 1
ma = moving_average(comments_frequency_diff, window_size)
plt.scatter(ma[:-(1+shift_size)].flatten(), diff_title_1[window_size+shift_size:].flatten(), s=0.05, alpha=1)

### B. Similarity

In [ ]:
comm_sim_list = []
diff_pair_freq_list = []
for i in range(len(diff_pair_freq)):
    for key, value in diff_pair_freq[i].items():
        comm_sim_list.append(comments_similarity[i][key[0], key[1]])
        diff_pair_freq_list.append(value)

In [ ]:
diff_pair_freq

In [ ]:
plt.scatter(comm_sim_list, diff_pair_freq_list, s=1, alpha=0.5)
plt.xlabel('Comments similarity (T)')
plt.ylabel('Frequency of different pairs in titles (T+1)')
corr = np.corrcoef(comm_sim_list, diff_pair_freq_list)[0][1]
print(corr)

In [ ]:
comm_sim_list = []
diff_pair_freq_list = []
for i in range(1, len(diff_pair_freq)):
    for key, value in diff_pair_freq[i].items():
        comm_sim_list.append(comments_similarity[i][key[0], key[1]] - comments_similarity[i-1][key[0], key[1]])
        diff_pair_freq_list.append(value)

In [ ]:
plt.scatter(comm_sim_list, diff_pair_freq_list, s=1, alpha=0.5)
plt.xlabel('Comments similarity difference (T-(T-1))')
plt.ylabel('Frequency of different pairs in titles (T+1)')
corr = np.corrcoef(comm_sim_list, diff_pair_freq_list)[0][1]
print(corr)

## 6. ETC